In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import plotly.express as px
import numpy as np
pd.options.mode.chained_assignment = None

In [ ]:
path_files = r"C:\Users\eduque\Fraunhofer\DeV-KopSys-2 - Globale ESA\04_methods_and_models\Sectoral_End-Use_Energy_Scenario\post_processed_results\energy_base_totals\xlsx_per_country"
files = [
    ("demands_MA_v2.xlsx", "csv_pypsa"),
    ("demands_CL_v2.xlsx", "csv_pypsa"),
    ("demands_EG_v2.xlsx", "csv_pypsa"),
    ("demands_ZA_v2.xlsx", "csv_pypsa")
]

dfs = []
for file, sheet in files:
    file_path = os.path.join(path_files, file)
    df = pd.read_excel(file_path, sheet_name=sheet)
    dfs.append(df)

# Concatenate all DataFrames into one
df = pd.concat(dfs, ignore_index=True)


In [ ]:
## ONLY TO EXPORT files to csv
# df.to_csv("output_csv/all_demands_combined.csv", index=False)

if 'scenario' in df.columns:
    for scenario in df['scenario'].dropna().unique():
        df_scenario = df[df['scenario'] == scenario]
        df_scenario.drop(columns="scenario",inplace=True)
        filename = f"energy_totals_{scenario}.csv"
        df_scenario.to_csv("output_csv/"+ filename, index=False)

In [ ]:
# Residential Balance

#set correct indexes
df.set_index(['scenario',"country"], inplace=True)

columns_residential = [
    "scenario",
    "country",
    "electricity", 
    "biomass", 
    "gas", 
    "liq. hydrocarbons"
]

dfs = []
for file, sheet in files:
    file_path = os.path.join(path_files, file)
    df = pd.read_excel(file_path, sheet_name=sheet)
    df= df.rename(columns={
        "electricity residential": "electricity",
        "residential oil": "liq. hydrocarbons",
        "residential biomass": "biomass",
        "residential gas": "gas"
        # "base_2020": "baseline_2019"
        }, inplace=False)
    df= df[columns_residential]
    # df = df[df['scenario'] != 'base']
    df = df.drop(1).reset_index(drop=True)
    dfs.append(df)

# Create a dictionary mapping country code to DataFrame
country_dfs = {df["country"].iloc[0].lower(): df for df in dfs}

# Example usage:
df_ma = country_dfs["ma"]
df_cl = country_dfs["cl"]
df_eg = country_dfs["eg"]
df_za = country_dfs["za"]

In [ ]:
# GRAPHICS OPTION B

scenario_order = ["base",'RF_2030', 'EL_2030', 'RF_2050', 'EL_2050'] 

# Colors for residential carriers
residential_colors = {
    'electricity': '#f58220', # Orange
    'biomass': '#b2d235', # Green
    'gas': '#179c7d', # Blue-grey
    'liq. hydrocarbons': '#1c3f52', # Dark blue
}
residential_cols = list(residential_colors.keys())

# Make sure scenario is categorical and ordered for all dfs
for df in [df_za, df_ma, df_cl, df_eg]:
    df['scenario'] = pd.Categorical(df['scenario'], categories=scenario_order, ordered=True)

def plot_residential_area(df, country_name, ax):
    df_sorted = df.sort_values('scenario')
    df_pivot = df_sorted.set_index('scenario')[residential_cols]
    
    colors = [residential_colors[col] for col in df_pivot.columns]
    df_pivot.plot.area(ax=ax, color=colors, linewidth=0, legend=False)
    
    ax.set_title(country_name, fontsize=14, weight='bold')
    ax.set_xlabel('Scenario')
    ax.set_ylabel('Total Energy Demand [TWh]')
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

fig, axs = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

plot_residential_area(df_cl, 'Chile', axs[0, 0])
plot_residential_area(df_eg, 'Egypt', axs[0, 1])
plot_residential_area(df_za, 'South Africa', axs[1, 0])
plot_residential_area(df_ma, 'Morocco', axs[1, 1])

fig.suptitle('Residential Final End Use Energy Demand', fontsize=25, weight='bold', y=1.05, x=0.52)

# Add legend outside right
handles, labels = axs[0,0].get_legend_handles_labels()
fig.legend(handles, labels, title='Residential Carrier', loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize='large', frameon=False)
fig.savefig("Residential_demand_plot.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
residential_cols = ['electricity', 'biomass', 'gas',
       'liq. hydrocarbons']
df_eg["sum"]= df_eg[residential_cols].sum(axis=1)

In [ ]:

# graphic option A
scenario_order = ["base",'RF_2030', 'EL_2030', 'RF_2050', 'EL_2050'] 

# Colors for residential carriers
residential_colors = {
    'electricity': '#f58220', # Orange
    'biomass': '#b2d235', # Green
    'gas': '#179c7d', # Blue-grey
    'liq. hydrocarbons': '#1c3f52', # Dark blue
}
residential_cols = list(residential_colors.keys())

# Make sure scenario is categorical and ordered for all dfs
for df in [df_za, df_ma, df_cl, df_eg]:
    df['scenario'] = pd.Categorical(df['scenario'], categories=scenario_order, ordered=True)

def plot_residential_bar(df, country_name, ax):
    df_sorted = df.sort_values('scenario')
    df_pivot = df_sorted.set_index('scenario')[residential_cols]
    colors = [residential_colors[col] for col in df_pivot.columns]
    df_pivot.plot.bar(ax=ax, color=colors, legend=False, stacked=True, width=0.7)
    ax.set_title(country_name, fontsize=14, weight='bold')
    ax.set_xlabel('Scenario')
    ax.set_ylabel('total end use energy demand [TWh]')
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

fig, axs = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

plot_residential_bar(df_cl, 'Chile', axs[0, 0])
plot_residential_bar(df_eg, 'Egypt', axs[0, 1])
plot_residential_bar(df_za, 'South Africa', axs[1, 0])
plot_residential_bar(df_ma, 'Morocco', axs[1, 1])

fig.suptitle('Residential Final End Use Energy Demand', fontsize=25, weight='bold', y=1.05, x=0.52)
handles, labels = axs[0,0].get_legend_handles_labels()
fig.legend(handles, labels, title='Carrier', loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize='large', frameon=False)
fig.savefig("Residential_demand_bar_plot.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Technologies for residential
path_file = r'C:\Users\eduque\Fraunhofer\DeV-KopSys-2 - Globale ESA\04_methods_and_models\Sectoral_End-Use_Energy_Scenario\graphics\input_data'
df = pd.read_excel(os.path.join(path_file, 'residential_all_280825.xlsx'))
df = df[~df['carrier'].isin(["Coal",'Heat', 'Renewable Heat (Solar & Geothermal)'])]
df.drop(columns= "Residential Total", inplace =True, axis=1)
df["carrier"].replace({
    'Oil': "liq. hydrocarbons",
    'Gas': "gas",
    'Electricity': "electricity", 
    'Hydrogen': "hydrogen", 
    'Biomass': "biomass"
    }, inplace=True)
df.fillna(0, inplace=True)
df

In [ ]:
# Technologies for residential
path_file = r'C:\Users\eduque\Fraunhofer\DeV-KopSys-2 - Globale ESA\04_methods_and_models\Sectoral_End-Use_Energy_Scenario\graphics\input_data'
df = pd.read_excel(os.path.join(path_file, 'residential_all_280825.xlsx'))
df = df[~df['carrier'].isin(["Coal",'Heat', 'Renewable Heat (Solar & Geothermal)'])]
df.drop(columns= "Residential Total", inplace =True, axis=1)
df["carrier"].replace({
    'Oil': "liq. hydrocarbons",
    'Gas': "gas",
    'Electricity': "electricity", 
    'Hydrogen': "hydrogen", 
    'Biomass': "biomass"
    }, inplace=True)
df.fillna(0, inplace=True)

df_residential = df.copy()

df_residential.rename(columns={
    'Residential Space Heating': 'space heating',
    'Residential Space Cooling': 'space cooling',
    'Residential Water Heating': 'water heating',
    'Residential Cooking': 'cooking',
    'Residential Electrical Appliances': 'electrical appliances',
}, inplace=True)

tech_cols = [
    'space heating',
    'space cooling',
    'water heating',
    'cooking',
    'electrical appliances'
]

df_long = df_residential.melt(
    id_vars=['country', 'scenario', 'carrier'],
    value_vars=tech_cols,
    var_name='technology',
    value_name='value'
)


df_long

In [ ]:
# Filter for space heating, electricity carrier, and scenario EL_2050
mask = (
    (df_long['technology'] == 'space heating') &
    (df_long['carrier'] == 'electricity') &
    (df_long['scenario'] == 'EL_2050')
)

# Aggregate electricity value for space heating in EL_2050 across all countries
el_space_heating = df_long[mask].groupby('country')['value'].sum()

# Aggregate total space heating demand (all carriers) for EL_2050 across all countries
total_mask = (
    (df_long['technology'] == 'space heating') &
    (df_long['scenario'] == 'EL_2050')
)
total_space_heating = df_long[total_mask].groupby('country')['value'].sum()

# Calculate share for each country
share_space_heating_el = (el_space_heating / total_space_heating * 100).round(1)

# Display result
print("Elektrifizierungsanteil für space heating im Szenario EL_2050 (%):")
print(share_space_heating_el)
print("Durchschnitt aller Länder: {:.1f} %".format(share_space_heating_el.mean()))

In [ ]:
# Aggregate all carriers for each technology, country, and scenario
df_agg = df_long.groupby(['country', 'scenario', 'technology'], as_index=False)['value'].sum()

# Calculate total value per country and scenario (across all technologies)
total_per_scenario = df_agg.groupby(['country', 'scenario'])['value'].transform('sum')

# Calculate share for each technology
df_agg['share'] = df_agg['value'] / total_per_scenario

# Create a pivot table for easier viewing (optional)
df_shares = df_agg.pivot_table(
    index=['country', 'scenario'],
    columns='technology',
    values='share'
).reset_index()

df_shares

In [ ]:
countries = df_long['country'].unique()
carrier_order = df_long["carrier"].unique()
scenario_order = df_long["scenario"].unique()
tech_order = tech_cols  # or sorted(df_long['technology'].unique())

residential_colors = {
    'liq. hydrocarbons': '#1c3f52',
    'electricity': '#f58220',
    'hydrogen': '#179c7d',
    'biomass': '#b2d235',
    'ammonia': '#fce356',
}

colors = [residential_colors.get(col, "#cccccc") for col in carrier_order]

for cty in countries:
    d = df_long[df_long['country'] == cty]
    fig, axs = plt.subplots(1, len(tech_order), figsize=(4*len(tech_order), 5), sharey=False)
    for i, tech in enumerate(tech_order):
        ax = axs[i]
        d_tech = d[d['technology'] == tech].copy()
        d_tech['scenario'] = pd.Categorical(d_tech['scenario'], categories=scenario_order, ordered=True)
        pivot = d_tech.pivot_table(index='scenario', columns='carrier', values='value', aggfunc='sum').fillna(0)
        for carrier in carrier_order:
            if carrier not in pivot.columns:
                pivot[carrier] = 0
        pivot = pivot[carrier_order]
        bottom = np.zeros(len(pivot.index))
        for idx, carrier in enumerate(carrier_order):
            values = pivot[carrier].values
            ax.bar(pivot.index, values, bottom=bottom, color=colors[idx], label=carrier)
            bottom += values
        ax.set_title(tech, fontsize=12, weight='bold')
        ax.set_xlabel('Scenario')
        if i == 0:
            ax.set_ylabel('Total end use energy demand [TWh]')
        ax.tick_params(axis='x', rotation=45)
    fig.suptitle(f"Residential energy by technology, scenario, and carrier — {cty}", fontsize=16, weight='bold')
    fig.legend(carrier_order, title="Carrier", loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize='small', frameon=False)
    plt.tight_layout()
    plt.show()

In [ ]:
# mode_order = ["road", "maritime (all)", "aviation (all)", "rail"]
# carrier_order = ['electricity', 'hydrogen', 'liq. hydrocarbons', 'biomass', 'ammonia']
# scenario_order = ['RF_2030', 'EL_2030', 'RF_2050', 'EL_2050']

# for cty, d in long_df_grouped_sum.groupby("country"):
#     full_name = country_dic.get(cty, cty)
#     fig, axs = plt.subplots(1, len(mode_order), figsize=(16, 5), sharey=False)
#     for i, mode in enumerate(mode_order):
#         ax = axs[i]
#         d_mode = d[d["mode_grouped"] == mode]
#         d_mode = d_mode.copy()
#         d_mode['scenario'] = pd.Categorical(d_mode['scenario'], categories=scenario_order, ordered=True)
#         pivot = d_mode.pivot_table(index='scenario', columns='carrier', values='value', aggfunc='sum').fillna(0)
#         bottom = None
#         for carrier in carrier_order:
#             values = pivot[carrier] if carrier in pivot.columns else [0]*len(pivot)
#             ax.bar(pivot.index, values, bottom=bottom, color=transport_colors.get(carrier, "#cccccc"), label=carrier)
#             bottom = values if bottom is None else bottom + values
#         ax.set_title(mode)
#         ax.set_xlabel('Scenario')
#         if i == 0:
#             ax.set_ylabel('Final end use energy demand (TWh)')
#         ax.tick_params(axis='x', rotation=45)
#     fig.suptitle(f"Final energy by mode, scenario, and carrier — {full_name}", fontsize=16, weight='bold')
#     fig.legend(carrier_order, title="Carrier", loc='center left', bbox_to_anchor=(1.0, 0.5))
#     plt.tight_layout()
#     plt.show()

# Transport Balance

In [ ]:

path_files = r"C:\Users\eduque\Fraunhofer\DeV-KopSys-2 - Globale ESA\04_methods_and_models\Sectoral_End-Use_Energy_Scenario\post_processed_results\energy_base_totals\xlsx_per_country"
files = [
    ("demands_MA_v2.xlsx", "csv_pypsa"),
    ("demands_CL_v2.xlsx", "csv_pypsa"),
    ("demands_EG_v2.xlsx", "csv_pypsa"),
    ("demands_ZA_v2.xlsx", "csv_pypsa")
]

dfs = []
for file, sheet in files:
    file_path = os.path.join(path_files, file)
    df = pd.read_excel(file_path, sheet_name=sheet)
    dfs.append(df)

# Concatenate all DataFrames into one
df = pd.concat(dfs, ignore_index=True)
df.set_index(['scenario',"country"], inplace=True)
df.drop(index = [("base_2030")], inplace=True)

In [ ]:
columns_transport = [
    "total road ev",
    "total road fcev",
    "total road ice",
    "road biomass",
    "total domestic aviation",
    "total domestic navigation",
    "total navigation hydrogen",
    "total navigation oil",
    "total navigation ammonia",
    "total navigation biomass",
    "total aviation biomass",
    "total international aviation",
    "total international navigation",
    "total rail hydrogen",
    "electricity rail",
    "total rail",
    # "international aviation e-fuel",
]

df_transport= df[columns_transport]

df_transport.rename(columns={
    "total road ev": "road electricity",
    "total road fcev": "road hydrogen",
    "total road ice": "road liq. hydrocarbon",
    "road biomass": "road biomass", # substract this from road hydrocarbon
    "total domestic aviation": "dom. aviation",
    "total domestic navigation": "dom. maritime",
    "total navigation hydrogen": "dom. maritime hydrogen",
    "total navigation oil":"dom. maritime hydrocarbon",
    "total navigation biomass": "dom. maritime biomass",
    "total aviation biomass": "dom. aviation biomass",
    "total navigation ammonia": "dom. maritime ammonia",
    "total international aviation": "int. aviation",
    "total international navigation": "int. maritime",
    "total rail hydrogen": "rail hydrogen",
    "electricity rail": "rail electricity",
    }, inplace=True)

# calculate correctly the road liquid hydrocarbon demand
df_transport["road liq. hydrocarbon"] = df_transport["road liq. hydrocarbon"] - df_transport["road biomass"]
# df_transport["dom. maritime hydrocarbon"] = df_transport["dom. maritime"] - df_transport["dom. maritime hydrogen"] - df_transport["dom. maritime ammonia"]
df_transport["rail liq. hydrocarbons"] = df_transport["total rail"] - df_transport["rail hydrogen"] - df_transport["rail electricity"]
df_transport["dom. aviation liq. hydrocarbon"] = df_transport["dom. aviation"] - df_transport["dom. aviation biomass"]
# drop all unnecessary columns
df_transport.drop(columns=["dom. maritime","total rail","dom. aviation"], inplace=True)


# Define mapping for each column: (mode, carrier)
column_map = {
    "road electricity": ("road", "electricity"),
    "road hydrogen": ("road", "hydrogen"),
    "road liq. hydrocarbon": ("road", "liq. hydrocarbons"),
    "road biomass": ("road", "biomass"),
    "dom. aviation biomass": ("aviation", "biomass"),
    "dom. aviation liq. hydrocarbon": ("aviation", "liq. hydrocarbons"),
    "int. aviation": ("int. aviation", "liq. hydrocarbons"),
    "dom. maritime hydrogen": ("maritime", "hydrogen"),
    "dom. maritime hydrocarbon": ("maritime", "liq. hydrocarbons"),
    "dom. maritime biomass": ("maritime", "biomass"),
    "dom. maritime ammonia": ("maritime", "ammonia"),
    "int. maritime": ("int. maritime", "liq. hydrocarbons"),
    "rail hydrogen": ("rail", "hydrogen"),
    "rail electricity": ("rail", "electricity"),
    "rail liq. hydrocarbons": ("rail", "liq. hydrocarbons"),
}


# Reindex columns to MultiIndex with mode first, carrier second
new_columns = []
for col in df_transport.columns:
    if col in column_map:
        new_columns.append(column_map[col])
    else:
        # If not mapped, keep original name as mode, and set carrier as 'Other'
        new_columns.append((col, "Other"))
df_transport.columns = pd.MultiIndex.from_tuples(new_columns)

# drop scenarios
df_transport = df_transport.drop(df_transport.index[df_transport.index.get_level_values("scenario") == "base"])


# Convert wide MultiIndex DataFrame to long format
long_df = (
    df_transport
    .stack(level=[0, 1])  # Stack mode and carrier columns
    .reset_index()        # Convert index to columns
    .rename(columns={
        0: "value",
        "level_2": "mode",
        "level_3": "carrier"})  # Rename the value column
)

# Prepare mode_grouped column with new categories for grouped plotting
mode_grouped_order = ["road", "rail", "aviation (all)", "maritime (all)"]
long_df_grouped = long_df.copy()
long_df_grouped["mode_grouped"] = long_df_grouped["mode"].astype(str)
long_df_grouped.loc[long_df_grouped["mode"].str.contains("aviation"), "mode_grouped"] = "aviation (all)"
long_df_grouped.loc[long_df_grouped["mode"].str.contains("maritime"), "mode_grouped"] = "maritime (all)"
long_df_grouped["mode_grouped"] = pd.Categorical(long_df_grouped["mode_grouped"], categories=mode_grouped_order, ordered=True)
long_df_grouped

# fix the several values with same indexes
long_df_grouped_sum = (
    long_df_grouped
    .groupby(["country", "scenario", "mode_grouped", "carrier"], as_index=False)["value"]
    .sum()
)



In [ ]:
transport_colors = {
    'liq. hydrocarbons': '#1c3f52',
    'electricity': '#f58220',
    'hydrogen': '#179c7d',
    'biomass': '#b2d235',
    'ammonia': '#fce356',
}
country_dic = {"CL":"Chile","EG":"Egypt","MA":"Morocco","ZA":"South Africa"}

def plot_per_country(df, facet_gap=0.08, ytick_size=12, fig_width=1100, fig_height=420, bargap=0.15):
    figs = {}
    mode_order = ["road", "maritime (all)", "aviation (all)", "rail"]
    carrier_order = ['electricity','hydrogen','liq.hydrocarbons','biomass','ammonia']
    scenario_order = ['RF_2030', 'EL_2030', 'RF_2050', 'EL_2050']

    for cty, d in df.groupby("country"):
        full_name = country_dic.get(cty, cty)
        fig = px.bar(
            d,
            x="scenario",
            y="value",
            color="carrier",
            facet_col="mode_grouped",
            facet_col_spacing=facet_gap,
            category_orders={"mode_grouped": mode_order, "carrier": carrier_order, "scenario": scenario_order},
            color_discrete_map=transport_colors,   # <-- custom colors here
            barmode="stack",
            labels={"value": "Final end use energy demand (TWh)", "carrier": "Carrier"},
            title=f"Final energy by mode, scenario, and carrier — {full_name}",
        )
        fig.update_yaxes(matches=None, showticklabels=True, ticks="outside",
                         automargin=True, tickfont=dict(size=ytick_size))
        fig.update_layout(bargap=bargap, width=fig_width, height=fig_height,
                          margin=dict(l=90, r=20, t=60, b=40), legend_title_text="Carrier")
        figs[cty] = fig

        # only one scenario x-axis title
        fig.update_xaxes(title_text=None)
        # fig.add_annotation(
        #     x=-0.1, y=-0.08, xref="paper", yref="paper",
        #     showarrow=False, text="Scenario"
        # )
        fig.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[-1]))
        fig.for_each_xaxis(lambda ax: ax.update(categoryorder="array", categoryarray=scenario_order))


    return figs

figs = plot_per_country(long_df_grouped_sum)

In [ ]:
# for key, fig in figs.items():
#     print(f"Figure: {key}")
#     fig.show()

In [ ]:

mode_order = ["road", "maritime (all)", "aviation (all)", "rail"]
carrier_order = ['electricity', 'hydrogen', 'liq. hydrocarbons', 'biomass', 'ammonia']
scenario_order = ['RF_2030', 'EL_2030', 'RF_2050', 'EL_2050']

for cty, d in long_df_grouped_sum.groupby("country"):
    full_name = country_dic.get(cty, cty)
    fig, axs = plt.subplots(1, len(mode_order), figsize=(16, 5), sharey=False)
    for i, mode in enumerate(mode_order):
        ax = axs[i]
        d_mode = d[d["mode_grouped"] == mode]
        d_mode = d_mode.copy()
        d_mode['scenario'] = pd.Categorical(d_mode['scenario'], categories=scenario_order, ordered=True)
        pivot = d_mode.pivot_table(index='scenario', columns='carrier', values='value', aggfunc='sum').fillna(0)
        bottom = None
        for carrier in carrier_order:
            values = pivot[carrier] if carrier in pivot.columns else [0]*len(pivot)
            ax.bar(pivot.index, values, bottom=bottom, color=transport_colors.get(carrier, "#cccccc"), label=carrier)
            bottom = values if bottom is None else bottom + values
        ax.set_title(mode)
        ax.set_xlabel('Scenario')
        if i == 0:
            ax.set_ylabel('Final end use energy demand (TWh)')
        ax.tick_params(axis='x', rotation=45)
    fig.suptitle(f"Final energy by mode, scenario, and carrier — {full_name}", fontsize=16, weight='bold')
    fig.legend(carrier_order, title="Carrier", loc='center left', bbox_to_anchor=(1.0, 0.5))
    plt.tight_layout()
    plt.show()

In [ ]:
# # Create a pivot table for analysis by mode, scenario, and carrier
# mode_analysis_table = long_df_grouped_sum.pivot_table(
#     index=['country', 'mode_grouped', 'scenario'],
#     columns='carrier',
#     values='value',
#     aggfunc='sum'
# ).reindex(columns=carrier_order).reset_index()

# # Optionally, rename country codes to full names
# mode_analysis_table['country'] = mode_analysis_table['country'].map(country_dic)

# # Display the table
# from IPython.display import display
# display(mode_analysis_table)

# # Optionally, export to Excel
# mode_analysis_table.to_excel("Transport_mode_analysis_table.xlsx", index=False)

In [ ]:
# Aggregate all modes for each country, scenario, and carrier
df_agg = (
    long_df_grouped
    .groupby(["country", "scenario", "carrier"], as_index=False)["value"]
    .sum()
)

df_agg

In [ ]:
# transport_colors = {
#     'liq. hydrocarbons': '#1c3f52',
#     'electricity': '#f58220',
#     'hydrogen': '#179c7d',
#     'biomass': '#b2d235',
#     'ammonia': '#fce356',
# }
# country_dic = {"CL": "Chile", "EG": "Egypt", "MA": "Morocco", "ZA": "South Africa"}
# carrier_order = ['Electricity', 'Hydrogen', 'Liq. hydrocarbons', 'Biomass', 'Ammonia']
# scenario_order = ['RF_2030', 'EL_2030', 'RF_2050', 'EL_2050']

# # Aggregate all modes for each country, scenario, and carrier
# df_agg = (
#     long_df_grouped
#     .groupby(["country", "scenario", "carrier"], as_index=False)["value"]
#     .sum()
# )

# # Map country codes to full names
# df_agg["country_full"] = df_agg["country"].map(country_dic)

# fig = px.bar(
#     df_agg,
#     x="country_full",
#     y="value",
#     color="carrier",
#     facet_col="scenario",
#     category_orders={"carrier": carrier_order, "scenario": scenario_order, "country_full": [country_dic[c] for c in country_dic]},
#     color_discrete_map=transport_colors,
#     barmode="stack",
#     labels={"value": "Final end use energy demand (TWh)", "carrier": "Carrier", "country_full": "Country"},
#     title="Final energy demand by country, scenario, and carrier",
# )

# # fig.update_yaxes(matches=None, showticklabels=True, ticks="outside",
# #                  automargin=True, tickfont=dict(size=12))
# fig.update_layout(bargap=0.15, width=1100, height=420,
#                   margin=dict(l=90, r=20, t=60, b=40), legend_title_text="Carrier")
# fig.update_xaxes(title_text=None)
# fig.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[-1]))
# fig.for_each_xaxis(lambda ax: ax.update(categoryorder="array", categoryarray=[country_dic[c] for c in country_dic]))

# fig.show()

In [ ]:
# Define color mapping for carriers
transport_colors = {
    'liq. hydrocarbons': '#1c3f52',
    'electricity': '#f58220',
    'hydrogen': '#179c7d',
    'biomass': '#b2d235',
    'ammonia': '#fce356',
}
carrier_order = ['electricity', 'hydrogen', 'liq. hydrocarbons', 'biomass', 'ammonia']
scenario_order = ['RF_2030', 'EL_2030', 'RF_2050', 'EL_2050']
country_dic = {"CL": "Chile", "EG": "Egypt", "MA": "Morocco", "ZA": "South Africa"}

def plot_transport_bar_agg(df_agg, country_code, ax):
    df_country = df_agg[df_agg['country'] == country_code].copy()
    df_country['scenario'] = pd.Categorical(df_country['scenario'], categories=scenario_order, ordered=True)
    df_pivot = df_country.pivot(index='scenario', columns='carrier', values='value')
    df_pivot = df_pivot[carrier_order]  # Ensure column order
    colors = [transport_colors[c] for c in df_pivot.columns]
    df_pivot.plot.bar(ax=ax, color=colors, width=0.7, legend=False, stacked=True)
    ax.set_title(country_dic.get(country_code, country_code), fontsize=14, weight='bold')
    ax.set_xlabel('Scenario')
    ax.set_ylabel('total end use energy demand [TWh]')
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

fig, axs = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

plot_transport_bar_agg(df_agg, 'CL', axs[0, 0])
plot_transport_bar_agg(df_agg, 'EG', axs[0, 1])
plot_transport_bar_agg(df_agg, 'ZA', axs[1, 0])
plot_transport_bar_agg(df_agg, 'MA', axs[1, 1])

fig.suptitle('Transport Final End Use Energy Demand', fontsize=25, weight='bold', y=1.05, x=0.52)

handles, labels = axs[0,0].get_legend_handles_labels()
fig.legend(handles, labels, title='Carrier', loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize='large', frameon=False)
fig.savefig("Transport_demand_bar_plot.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# # Create a pivot table for the analysis table
# analysis_table = df_agg.pivot_table(
#     index=['country', 'scenario'],
#     columns='carrier',
#     values='value',
#     aggfunc='sum'
# ).reindex(columns=carrier_order).reset_index()

# # Optionally, rename country codes to full names
# analysis_table['country'] = analysis_table['country'].map(country_dic)

# # Display the table
# print(analysis_table)

# # If you want a nicely formatted table in the notebook:
# import pandas as pd
# from IPython.display import display

# display(analysis_table)

# analysis_table.to_excel("Transport_demand_analysis_table.xlsx", index=False)

In [ ]:

# # Create subplot grid
# fig = make_subplots(rows=2, cols=2, subplot_titles=[country_dic[c] for c in ['CL', 'EG', 'ZA', 'MA']])

# country_positions = {'CL': (1, 1), 'EG': (1, 2), 'ZA': (2, 1), 'MA': (2, 2)}
# carrier_order = ['electricity', 'hydrogen', 'liq. hydrocarbons', 'biomass', 'ammonia']

# for country_code, (row, col) in country_positions.items():
#     df_country = df_agg[df_agg['country'] == country_code].copy()
#     df_country['scenario'] = pd.Categorical(df_country['scenario'], categories=scenario_order, ordered=True)
#     for carrier in carrier_order:
#         df_c = df_country[df_country['carrier'] == carrier]
#         fig.add_trace(
#             go.Bar(
#                 x=df_c['scenario'],
#                 y=df_c['value'],
#                 name=carrier,
#                 marker_color=transport_colors[carrier],
#                 showlegend=(row == 1 and col == 1),  # Show legend only once
                
#             ),
#             row=row, col=col
#         )

# fig.update_layout(
#     barmode= "stack",
#     height=700, width=1200,
#     title_x=0.5,
#     title_text="Transport Final End Use Energy Demand",
#     legend_title_text="Carrier",
#     title_font=dict(size=25, family="Arial", color="black", weight="bold")
# )

# fig.update_xaxes(showline=True, linewidth=2, linecolor='black')
# fig.update_yaxes(showline=True, linewidth=2, linecolor='black')


# fig.update_yaxes(title_text="total end use demand [TWH]")

fig.show()

In [ ]:
# STACKED GRAPHIC
# # Define color mapping for carriers
# transport_colors = {
#     'liq. hydrocarbons': '#1c3f52',
#     'electricity': '#f58220',
#     'hydrogen': '#179c7d',
#     'biomass': '#b2d235',
#     'ammonia': '#fce356',
# }
# carrier_order = ['electricity', 'hydrogen', 'liq. hydrocarbons', 'biomass', 'ammonia']
# scenario_order = ['RF_2030', 'EL_2030', 'RF_2050', 'EL_2050']
# country_dic = {"CL": "Chile", "EG": "Egypt", "MA": "Morocco", "ZA": "South Africa"}

# # Aggregate all modes for each country, scenario, and carrier
# df_agg = (
#     long_df_grouped
#     .groupby(["country", "scenario", "carrier"], as_index=False)["value"]
#     .sum()
# )

# def plot_transport_area_agg(df_agg, country_code, ax):
#     df_country = df_agg[df_agg['country'] == country_code].copy()
#     df_country['scenario'] = pd.Categorical(df_country['scenario'], categories=scenario_order, ordered=True)
#     df_pivot = df_country.pivot(index='scenario', columns='carrier', values='value')
#     df_pivot = df_pivot[carrier_order]  # Ensure column order
#     colors = [transport_colors[c] for c in df_pivot.columns]
#     df_pivot.plot.area(ax=ax, color=colors, linewidth=0, legend=False)
#     ax.set_title(country_dic.get(country_code, country_code), fontsize=14, weight='bold')
#     ax.set_xlabel('Scenario')
#     ax.set_ylabel('Total Energy Demand [TWh]')
#     ax.grid(alpha=0.3)
#     ax.tick_params(axis='x', rotation=45)

# fig, axs = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

# plot_transport_area_agg(df_agg, 'CL', axs[0, 0])
# plot_transport_area_agg(df_agg, 'EG', axs[0, 1])
# plot_transport_area_agg(df_agg, 'ZA', axs[1, 0])
# plot_transport_area_agg(df_agg, 'MA', axs[1, 1])

# fig.suptitle('Transport Final End Use Energy Demand', fontsize=25, weight='bold', y=1.05, x=0.52)

# handles, labels = axs[0,0].get_legend_handles_labels()
# fig.legend(handles, labels, title='Transport Carrier', loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize='small', frameon=False)
# fig.savefig("Transport_demand_plot.png", dpi=300, bbox_inches='tight')
# plt.show()

# Indusrty Balance

In [ ]:

path_files = r"C:\Users\eduque\Fraunhofer\DeV-KopSys-2 - Globale ESA\04_methods_and_models\Sectoral_End-Use_Energy_Scenario\post_processed_results\industry_demand\xlsx_per_country"

files = [
    ("industry_CL.xlsx", "csv_pypsa"),
    ("industry_EG.xlsx", "csv_pypsa"),
    ("industry_MA.xlsx", "csv_pypsa"),
    ("industry_ZA.xlsx", "csv_pypsa")
]

dfs = []
for file, sheet in files:
    file_path = os.path.join(path_files, file)
    df = pd.read_excel(file_path, sheet_name=sheet, engine='openpyxl',header=2)
    df.drop(columns= "year", inplace=True)
    df["carrier"] = df["carrier"].replace("oil","liq. hydrocarbons")
    df = df[~df["carrier"].isin(["process emissions", "low-temperature heat"])]
    df = df[~df["scenario"].isin(["base","baseline_2030","baseline_2050"])]
    df = df.fillna(0)
    dfs.append(df)

# Concatenate all DataFrames into one and clean up
df = pd.concat(dfs, ignore_index=True)


In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import plotly.express as px
import numpy as np
pd.options.mode.chained_assignment = None

In [15]:

# path_files = r"C:\Users\eduque\Desktop\Morocco_paper\demands\industry_totals"

path_files = r"C:\Users\eduque\Fraunhofer\DeV-KopSys-2 - Globale ESA\04_methods_and_models\Sectoral_End-Use_Energy_Scenario\post_processed_results\industry_demand\MA_paper"
files = [
    ("industry_MA_paper.xlsx", "csv_pypsa"),
#     ("industry_EG.xlsx", "csv_pypsa"),
#     ("industry_MA.xlsx", "csv_pypsa"),
#     ("industry_ZA.xlsx", "csv_pypsa")
 ]

dfs = []
for file, sheet in files:
    file_path = os.path.join(path_files, file)
    df = pd.read_excel(file_path, sheet_name=sheet, engine='openpyxl',header=2)
    df.drop(columns= "year", inplace=True)
    # df["carrier"] = df["carrier"].replace("oil","liq. hydrocarbons")
    # df = df[~df["carrier"].isin(["process emissions", "low-temperature heat"])]
    df = df[~df["scenario"].isin(["base","baseline_2030","baseline_2050"])]
    df = df.fillna(0)
    dfs.append(df)

# Concatenate all DataFrames into one and clean up
df = pd.concat(dfs, ignore_index=True)


In [16]:
## ONLY TO EXPORT files to csv
# df.to_csv("output_csv/all_demands_combined.csv", index=False)
if 'scenario' in df.columns:
    for scenario in df['scenario'].dropna().unique():
        df_scenario = df[df['scenario'] == scenario]
        df_scenario.drop(columns="scenario",inplace=True)
        filename = f"industry_totals_{scenario}.csv"
        df_scenario.to_csv("output_csv_MA/"+ filename, index=False)

In [ ]:
# prepare df:
df_long = df.melt(
    id_vars=["country", "scenario", "carrier"],  
    var_name="industry",                         
    value_name="value"                         
)

agg_col =[   
    "construction",
    "food and tobacco",
    "machinery",
    "other",
    "*other",
    "textile and leather",
    "transport equipment",
    "wood and wood products"
]

# atl
# industries_to_show = [
#     "chemical and petrochemical",
#     #"construction",
#     #"food and tobacco",
#     "iron and steel",
#     #"machinery",
#     "mining and quarrying",
#     "non-ferrous metals",
#     "non-metallic minerals",
#    # "other",
#     "paper pulp and print",
#     #"textile and leather",
#     #"transport equipment",
#     #"wood and wood products"
# ]
# df_long = df_long[df_long["industry"].isin(industries_to_show)]


# Filter and sum the rows to aggregate
all_others_df = df_long[df_long["industry"].isin(agg_col)].groupby(
    ["country", "scenario", "carrier"]
)["value"].sum().reset_index()
all_others_df["industry"] = "all_others"

# Remove original industries and add the aggregated version
df_long_agg = pd.concat([
    df_long[~df_long["industry"].isin(agg_col)],
    all_others_df
], ignore_index=True)

df_long_agg["value"] = df_long_agg["value"]/1000000 # from MWh to TWh


In [ ]:
industry_colors = {
    'chemical and petrochemical': '#1c3f52',
    #'construction': '#f58220',
    #'food and tobacco': '#179c7d',
    'iron and steel': '#b2d235',
    #'machinery': '#fce356',
    'non-ferrous metals': '#a6bbc8',
    'non-metallic minerals': '#008598',
    'mining and quarrying': '#e377c2',
    #'other': '#FCD80E',
    'paper pulp and print': '#228b22',
    "all_others":"#FCD80E",
    #'textile and leather': '#d62728',
    #'transport equipment': '#56fcc5',
    #'wood and wood products': '#b2d235',
}
# Colors for industry carriers
industry_carrier_colors = {
    'liq. hydrocarbons': '#1c3f52',
    'electricity': '#f58220',
    'hydrogen': '#179c7d',
    'biomass': '#b2d235',
    'ammonia': '#fce356',
    'gas': '#a6bbc8',
    'coal': '#454545',
}

facet_name_map = {
    "chemical and petrochemical": "petrochemicals",
    #"construction",
    #"food and tobacco",
    "iron and steel": "iron and steel",
    #"machinery",
    "mining and quarrying": "mining",
    "non-ferrous metals" :"non-ferrous metals",
    "non-metallic minerals": "non-metallic minerals",
    #"other",
    "all_others":"others",
    "paper pulp and print": "paper and pulp",
    #"textile and leather",
    #"transport equipment",
    #"wood and wood products"
}
# industry_cols = list(industry_colors.keys())

country_dic = {"CL": "Chile", "EG": "Egypt", "MA": "Morocco", "ZA": "South Africa"}
industry_order = list(industry_colors.keys())
scenario_order = ['RF_2030', 'EL_2030', 'RF_2050', 'EL_2050']

def plot_per_country_industry(df, facet_gap=0.08, ytick_size=12, fig_width=1100, fig_height=420, bargap=0.15):
    figs = {}
    for cty, d in df.groupby("country"):
        full_name = country_dic.get(cty, cty)
        fig = px.bar(
            d,
            x="scenario",
            y="value",
            color="carrier",
            facet_col="industry",
            facet_col_spacing=facet_gap,
            category_orders={"industry": industry_order, "scenario": scenario_order},
            color_discrete_map=industry_carrier_colors,
            barmode="stack",
            labels={"value": "Final end use energy demand (TWh)", "industry": "Industry"},
            title=f"Final energy by industry, scenario, and carrier — {full_name}",
        )
        # Update facet labels using the dictionary
        def short_label(a):
            label = a.text.split("=")[-1].strip()  # Remove prefix and strip whitespace
            label = facet_name_map.get(label, label)
            a.update(text=label)
        fig.for_each_annotation(short_label)
        fig.update_yaxes(matches=None, showticklabels=True, ticks="outside",
                         automargin=True, tickfont=dict(size=ytick_size))
        fig.update_layout(bargap=bargap, width=fig_width, height=fig_height,
                          margin=dict(l=90, r=20, t=60, b=40), legend_title_text="Carrier")
        figs[cty] = fig

        fig.update_xaxes(title_text=None)
        fig.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[-1]))
        fig.for_each_xaxis(lambda ax: ax.update(categoryorder="array", categoryarray=scenario_order))
    return figs

# Example usage:
figs = plot_per_country_industry(df_long_agg)

In [ ]:
for key, fig in figs.items():
    print(f"Figure: {key}")
    fig.show()

In [ ]:
industry_order = list(industry_colors.keys())
carrier_order = list(industry_carrier_colors.keys())
scenario_order = ['RF_2030', 'EL_2030', 'RF_2050', 'EL_2050']

for cty, d in df_long_agg.groupby("country"):
    full_name = country_dic.get(cty, cty)
    n_industries = len(industry_order)
    fig, axs = plt.subplots(1, n_industries, figsize=(4*n_industries, 5), sharey=False)
    for i, industry in enumerate(industry_order):
        ax = axs[i]
        d_ind = d[d["industry"] == industry]
        d_ind['scenario'] = pd.Categorical(d_ind['scenario'], categories=scenario_order, ordered=True)
        pivot = d_ind.pivot_table(index='scenario', columns='carrier', values='value', aggfunc='sum', orderned= False).fillna(0)
        # Ensure all carriers are present
        for carrier in carrier_order:
            if carrier not in pivot.columns:
                pivot[carrier] = 0
        pivot = pivot[carrier_order]
        bottom = np.zeros(len(pivot.index))
        for carrier in carrier_order:
            values = pivot[carrier].values
            ax.bar(pivot.index, values, bottom=bottom, color=industry_carrier_colors[carrier], label=carrier)
            bottom += values
        # Facet label
        ax.set_title(facet_name_map.get(industry, industry), fontsize=12, weight='bold')
        ax.set_xlabel('Scenario')
        if i == 0:
            ax.set_ylabel('Final end use energy demand (TWh)')
        ax.tick_params(axis='x', rotation=45)
    fig.suptitle(f"Final energy by industry, scenario, and carrier — {full_name}", fontsize=16, weight='bold')
    fig.legend(carrier_order, title="Carrier", loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize='small', frameon=False)
    plt.tight_layout()
    plt.show()

In [ ]:
# Create a pivot table for industry types analysis
industry_type_table = df_long_agg.pivot_table(
    index=['country', 'industry', 'scenario'],
    columns='carrier',
    values='value',
    aggfunc='sum'
).reindex(columns=carrier_order).reset_index()

# Optionally, map country codes to full names
industry_type_table['country'] = industry_type_table['country'].map(country_dic)

# Display the table
from IPython.display import display
display(industry_type_table)

# Optionally, export to Excel
industry_type_table.to_excel("Industry_type_analysis_table.xlsx", index=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

industry_order = list(industry_colors.keys())
carrier_order = list(industry_carrier_colors.keys())
scenario_order = ['RF_2030', 'EL_2030', 'RF_2050', 'EL_2050']
countries = list(country_dic.keys())

fig, axs = plt.subplots(len(countries), len(industry_order), figsize=(4*len(industry_order), 5*len(countries)), sharey=False)

for row, cty in enumerate(countries):
    full_name = country_dic.get(cty, cty)
    d = df_long_agg[df_long_agg["country"] == cty]
    for col, industry in enumerate(industry_order):
        ax = axs[row, col]
        d_ind = d[d["industry"] == industry]
        d_ind['scenario'] = pd.Categorical(d_ind['scenario'], categories=scenario_order, ordered=True)
        pivot = d_ind.pivot_table(index='scenario', columns='carrier', values='value', aggfunc='sum').fillna(0)
        for carrier in carrier_order:
            if carrier not in pivot.columns:
                pivot[carrier] = 0
        pivot = pivot[carrier_order]
        bottom = np.zeros(len(pivot.index))
        for carrier in carrier_order:
            values = pivot[carrier].values
            ax.bar(pivot.index, values, bottom=bottom, color=industry_carrier_colors[carrier], label=carrier)
            bottom += values
        if col == 0:
            ax.set_ylabel(full_name, fontsize=14, weight='bold')
        if row == 0:
            ax.set_title(facet_name_map.get(industry, industry), fontsize=12, weight='bold')
        # ax.set_xlabel('Scenario')
        
        ax.tick_params(axis='x', rotation=45)

fig.suptitle("Final energy by industry, scenario, and carrier", fontsize=18, weight='bold', y=1.02)
fig.legend(carrier_order, title="Carrier", loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize='large', frameon=False)
fig.legend(carrier_order, title="Carrier", loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize='large', frameon=False)
# fig.update(ax.set_ylabel('total end use energy demand [TWh]'))
plt.tight_layout()
plt.show()

In [ ]:
df_agg

In [ ]:
# Define color mapping for industry carriers
industry_carrier_colors = {
    'liq. hydrocarbons': '#1c3f52',
    'electricity': '#f58220',
    'hydrogen': '#179c7d',
    'biomass': '#b2d235',
    'ammonia': '#fce356',
    'gas': '#a6bbc8',
    'coal': '#454545',
}
carrier_order = ['electricity', 'hydrogen', 'liq. hydrocarbons', 'biomass', 'ammonia', 'gas', 'coal']
scenario_order = ['RF_2030', 'EL_2030', 'RF_2050', 'EL_2050']
country_dic = {"CL": "Chile", "EG": "Egypt", "MA": "Morocco", "ZA": "South Africa"}

def plot_industry_bar_agg(df_agg, country_code, ax):
    df_country = df_agg[df_agg['country'] == country_code].copy()
    df_country['scenario'] = pd.Categorical(df_country['scenario'], categories=scenario_order, ordered=True)
    df_pivot = df_country.pivot(index='scenario', columns='carrier', values='value')
    df_pivot = df_pivot.reindex(columns=carrier_order)  # Ensure column order
    colors = [industry_carrier_colors.get(c, "#cccccc") for c in df_pivot.columns]
    df_pivot.plot.bar(ax=ax, color=colors, width=0.7, legend=False, stacked=True)
    ax.set_title(country_dic.get(country_code, country_code), fontsize=14, weight='bold')
    ax.set_xlabel('Scenario')
    ax.set_ylabel('total end use energy demand [TWh]')
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

fig, axs = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

plot_industry_bar_agg(df_agg, 'CL', axs[0, 0])
plot_industry_bar_agg(df_agg, 'EG', axs[0, 1])
plot_industry_bar_agg(df_agg, 'ZA', axs[1, 0])
plot_industry_bar_agg(df_agg, 'MA', axs[1, 1])

fig.suptitle('Industry Final End Use Energy Demand', fontsize=25, weight='bold', y=1.05, x=0.52)

handles, labels = axs[0,0].get_legend_handles_labels()
fig.legend(handles, labels, title='Carrier', loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize='large', frameon=False)
fig.savefig("Industry_demand_bar_plot.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# for cty, d in df_agg.groupby("country"):
#     full_name = country_dic.get(cty, cty)
#     fig = px.bar(
#         d,
#         x="scenario",
#         y="value",
#         color="carrier",
#         category_orders={"scenario": scenario_order},
#         color_discrete_map=industry_carrier_colors,
#         barmode="stack",
#         labels={"value": "Final end use energy demand (TWh)", "carrier": "Carrier"},
#         title=f"Final energy by carrier and scenario — {full_name}",
#     )
#     fig.show()